# SAE + POD latente + SINDy

Notebook autocontido inspirado no artigo **Data discovery of low dimensional fluid dynamics of turbulent flows** (`arXiv:2401.05449`).

Ideia aplicada aqui:

```text
campo CFD -> SAE/Autoencoder -> latentes z(t) -> POD nos latentes -> coordenadas b(t) -> SINDy -> db/dt=f(b)
```

Diferença em relação aos notebooks anteriores:

- `POD_SINDy`: aplica POD direto no campo físico.
- `SAE_SINDy`: aplica SINDy direto nos latentes do autoencoder.
- **Este notebook**: aplica POD nos latentes do SAE para estabilizar/filtrar o espaço não linear antes do SINDy.


## 1. Importações e caminhos


In [ ]:
# Esta célula importa bibliotecas e força backend não-interativo para evitar erro do matplotlib_inline.
from __future__ import annotations

import csv
import json
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use("Agg", force=True)
matplotlib.rcParams["backend"] = "Agg"
import matplotlib.pyplot as plt
plt.switch_backend("Agg")

import numpy as np
from scipy.integrate import solve_ivp
from scipy.signal import savgol_filter
from sklearn.linear_model import Lasso
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SRC_DIR = Path(r"/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/SINDy/src").resolve()
CASE_DIR = SRC_DIR.parent.parent
OUTPUT_ROOT = CASE_DIR / "SINDy" / "resultados" / "sae_pod_sindy"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"SRC_DIR     = {SRC_DIR}")
print(f"CASE_DIR    = {CASE_DIR}")
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}")


def force_matplotlib_agg() -> None:
    # Garante que todas as figuras sejam salvas em arquivo sem depender do backend inline do Jupyter.
    matplotlib.use("Agg", force=True)
    matplotlib.rcParams["backend"] = "Agg"
    try:
        plt.switch_backend("Agg")
    except Exception:
        pass


## 2. Estruturas de dados e carregamento dos latentes SAE


In [ ]:
# Classe de dados: guarda as coordenadas reduzidas usadas pelo SINDy.
@dataclass
class ReducedDataset:
    name: str
    state_symbol: str
    states: np.ndarray
    times: np.ndarray
    source: Path
    metadata: dict[str, Any] = field(default_factory=dict)


# Classe de configuração da busca SAE-POD-SINDy.
@dataclass
class SearchConfig:
    latent_pod_modes_list: tuple[int, ...] = (2, 3, 4)
    polynomial_orders: tuple[int, ...] = (1, 2)
    use_sine_options: tuple[bool, ...] = (False,)
    regression_methods: tuple[str, ...] = ("stlsq", "lasso")
    alpha_values: tuple[float, ...] = (1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1)
    smooth_windows: tuple[int, ...] = (0, 21, 51)
    max_plot_states: int = 6
    trajectory_error_limit: float = 0.05
    derivative_error_limit: float = 0.50
    sparsity_weight: float = 0.25


# Função auxiliar: erro relativo L2 entre referência e estimativa.
def relative_l2(reference: np.ndarray, estimate: np.ndarray) -> float:
    return float(np.linalg.norm(reference - estimate) / (np.linalg.norm(reference) + 1e-12))


# Função auxiliar: converte NumPy para JSON.
def make_json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {key: make_json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


# Função auxiliar: ordena no tempo e remove duplicatas.
def sort_and_deduplicate(times: np.ndarray, states: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(times, kind="stable")
    times = np.asarray(times[order], dtype=np.float64)
    states = np.asarray(states[order], dtype=np.float64)
    unique_times, unique_idx = np.unique(times, return_index=True)
    if len(unique_times) != len(times):
        times = times[unique_idx]
        states = states[unique_idx]
    return times, states


# Função auxiliar: estima passo temporal típico.
def infer_dt(times: np.ndarray) -> float:
    deltas = np.diff(times)
    return 0.0 if deltas.size == 0 else float(np.median(deltas))


# Função de carregamento SAE: lê latents.npz e retorna z(t).
def load_sae_latents(case_dir: Path) -> ReducedDataset:
    candidates = (
        Path(case_dir) / "SAE" / "resultados" / "latents.npz",
        Path(case_dir) / "SAE" / "resultados" / "SAE" / "latents.npz",
    )
    latent_path = next((path for path in candidates if path.exists()), None)
    if latent_path is None:
        checked = "\n".join(str(path) for path in candidates)
        raise FileNotFoundError(f"Latentes SAE não encontrados. Caminhos testados:\n{checked}")

    data = np.load(latent_path, allow_pickle=True)
    if {"z_train", "z_val", "t_train", "t_val"}.issubset(set(data.files)):
        states = np.vstack([np.asarray(data["z_train"], dtype=np.float64), np.asarray(data["z_val"], dtype=np.float64)])
        times = np.concatenate([np.asarray(data["t_train"], dtype=np.float64), np.asarray(data["t_val"], dtype=np.float64)])
    elif "latent_data" in data.files:
        states = np.asarray(data["latent_data"], dtype=np.float64)
        times = np.asarray(data["times"], dtype=np.float64) if "times" in data.files else np.arange(states.shape[0], dtype=np.float64)
    else:
        raise KeyError("Formato de latents.npz não suportado.")

    times, states = sort_and_deduplicate(times, states)
    return ReducedDataset(
        name="sae_latent",
        state_symbol="z",
        states=states,
        times=times,
        source=latent_path,
        metadata={"latent_dim": int(states.shape[1]), "dt": infer_dt(times)},
    )


## 3. POD aplicado no espaço latente do SAE


In [ ]:
# Classe de transformação SAE -> POD latente -> SAE.
# Ela normaliza os latentes, aplica POD/SVD e permite reconstruir os latentes originais.
@dataclass
class LatentPODTransform:
    z_mean: np.ndarray
    z_std: np.ndarray
    pod_mean: np.ndarray
    components: np.ndarray
    singular_values: np.ndarray
    energy_capture: float
    q_mean: np.ndarray
    q_std: np.ndarray

    # Converte latentes SAE z(t) para coordenadas POD latentes q(t), já escaladas para SINDy.
    def transform(self, z: np.ndarray) -> np.ndarray:
        z = np.asarray(z, dtype=np.float64)
        z_norm = (z - self.z_mean[None, :]) / self.z_std[None, :]
        q = (z_norm - self.pod_mean[None, :]) @ self.components.T
        return (q - self.q_mean[None, :]) / self.q_std[None, :]

    # Converte coordenadas POD latentes escaladas de volta para latentes SAE originais.
    def inverse_transform(self, q_scaled: np.ndarray) -> np.ndarray:
        q_scaled = np.asarray(q_scaled, dtype=np.float64)
        q = q_scaled * self.q_std[None, :] + self.q_mean[None, :]
        z_norm = q @ self.components + self.pod_mean[None, :]
        return z_norm * self.z_std[None, :] + self.z_mean[None, :]


# Função principal desta etapa: aplica POD nos latentes z(t) do SAE.
def fit_latent_pod(dataset: ReducedDataset, n_modes: int) -> tuple[ReducedDataset, LatentPODTransform]:
    z = np.asarray(dataset.states, dtype=np.float64)

    # Passo 1: normaliza cada dimensão latente para média zero e desvio unitário.
    z_mean = np.mean(z, axis=0)
    z_std = np.std(z, axis=0)
    z_std[z_std < 1e-12] = 1.0
    z_norm = (z - z_mean[None, :]) / z_std[None, :]

    # Passo 2: centraliza novamente e aplica SVD/POD no espaço latente normalizado.
    pod_mean = np.mean(z_norm, axis=0)
    z_centered = z_norm - pod_mean[None, :]
    U, s, Vt = np.linalg.svd(z_centered, full_matrices=False)

    # Passo 3: seleciona os modos POD latentes.
    r = min(max(1, int(n_modes)), Vt.shape[0])
    components = Vt[:r, :]
    q = z_centered @ components.T
    energy_capture = float(np.sum(s[:r] ** 2) / (np.sum(s**2) + 1e-12))

    # Passo 4: escala as coordenadas q para melhorar o condicionamento do SINDy.
    q_mean = np.mean(q, axis=0)
    q_std = np.std(q, axis=0)
    q_std[q_std < 1e-12] = 1.0
    q_scaled = (q - q_mean[None, :]) / q_std[None, :]

    transform = LatentPODTransform(
        z_mean=z_mean,
        z_std=z_std,
        pod_mean=pod_mean,
        components=components,
        singular_values=s,
        energy_capture=energy_capture,
        q_mean=q_mean,
        q_std=q_std,
    )

    reduced = ReducedDataset(
        name="sae_pod_latent",
        state_symbol="b",
        states=q_scaled,
        times=dataset.times,
        source=dataset.source,
        metadata={
            "latent_dim_original": int(z.shape[1]),
            "latent_pod_modes": int(r),
            "latent_pod_energy_capture": energy_capture,
            "dt": infer_dt(dataset.times),
        },
    )
    return reduced, transform


# Função opcional: suaviza as coordenadas antes de derivar.
def smooth_states(states: np.ndarray, window_length: int) -> np.ndarray:
    if window_length is None or window_length <= 2:
        return np.asarray(states, dtype=np.float64)
    window = int(window_length)
    if window % 2 == 0:
        window += 1
    if window >= states.shape[0]:
        window = states.shape[0] - 1 if states.shape[0] % 2 == 0 else states.shape[0]
    if window < 5:
        return np.asarray(states, dtype=np.float64)
    return savgol_filter(states, window_length=window, polyorder=3, axis=0)


## 4. Biblioteca SINDy e regressão esparsa


In [ ]:
# Função da biblioteca SINDy: gera potências polinomiais até a ordem escolhida.
def generate_powers(n_variables: int, order: int) -> list[tuple[int, ...]]:
    if n_variables == 1:
        return [(power,) for power in range(order + 1)]
    powers: list[tuple[int, ...]] = []
    for power in range(order + 1):
        for suffix in generate_powers(n_variables - 1, order - power):
            powers.append((power,) + suffix)
    return sorted(powers, reverse=False)


# Classe da biblioteca SINDy: monta Theta(b) com termos polinomiais e opcionalmente seno/cosseno.
@dataclass
class PolynomialLibrary:
    n_variables: int
    order: int
    use_sine: bool
    variable_symbol: str

    def __post_init__(self) -> None:
        self.powers = generate_powers(self.n_variables, self.order)
        self.feature_names = self._build_feature_names()

    def _build_feature_names(self) -> list[str]:
        names = []
        for power in self.powers:
            if sum(power) == 0:
                names.append("1")
                continue
            parts = []
            for index, exponent in enumerate(power):
                if exponent == 0:
                    continue
                variable = f"{self.variable_symbol}{index + 1}"
                parts.append(variable if exponent == 1 else f"{variable}^{exponent}")
            names.append("*".join(parts))
        if self.use_sine:
            for index in range(self.n_variables):
                variable = f"{self.variable_symbol}{index + 1}"
                names.append(f"sin({variable})")
                names.append(f"cos({variable})")
        return names

    def transform(self, states: np.ndarray) -> np.ndarray:
        states = np.asarray(states, dtype=np.float64)
        theta = np.empty((states.shape[0], len(self.feature_names)), dtype=np.float64)
        for column, power in enumerate(self.powers):
            theta[:, column] = np.prod(np.power(states, power), axis=1)
        next_column = len(self.powers)
        if self.use_sine:
            for index in range(self.n_variables):
                theta[:, next_column] = np.sin(states[:, index])
                theta[:, next_column + 1] = np.cos(states[:, index])
                next_column += 2
        return theta

    def transform_single(self, state: np.ndarray) -> np.ndarray:
        return self.transform(np.asarray(state, dtype=np.float64)[None, :])[0]


# Função de derivadas: calcula db/dt.
def compute_derivatives(states: np.ndarray, times: np.ndarray) -> np.ndarray:
    return np.gradient(states, times, axis=0, edge_order=2 if len(times) > 2 else 1)


# Regressão Lasso simples, igual aos notebooks anteriores.
def fit_lasso(theta: np.ndarray, target: np.ndarray, alpha: float) -> np.ndarray:
    scale = np.linalg.norm(theta, axis=0)
    scale[scale < 1e-12] = 1.0
    theta_scaled = theta / scale[None, :]
    model = Lasso(alpha=alpha, fit_intercept=False, max_iter=50000, tol=1e-6)
    model.fit(theta_scaled, target)
    return model.coef_ / scale


# Regressão STLSQ/TLSA: mínimos quadrados com limiarização iterativa.
# Este método é mais próximo da literatura SINDy clássica e do espírito dos artigos discutidos.
def fit_stlsq(theta: np.ndarray, derivatives: np.ndarray, threshold: float, max_iter: int = 20) -> np.ndarray:
    Xi, *_ = np.linalg.lstsq(theta, derivatives, rcond=None)
    for _ in range(max_iter):
        small = np.abs(Xi) < threshold
        Xi[small] = 0.0
        for state_index in range(derivatives.shape[1]):
            active = ~small[:, state_index]
            if np.any(active):
                Xi[active, state_index], *_ = np.linalg.lstsq(theta[:, active], derivatives[:, state_index], rcond=None)
            else:
                Xi[:, state_index] = 0.0
    return Xi


# Função que ajusta todas as equações para um método/alpha.
def fit_sindy(theta: np.ndarray, derivatives: np.ndarray, method: str, alpha: float) -> np.ndarray:
    if method == "stlsq":
        return fit_stlsq(theta, derivatives, threshold=alpha)
    if method == "lasso":
        Xi = np.zeros((theta.shape[1], derivatives.shape[1]), dtype=np.float64)
        for state_index in range(derivatives.shape[1]):
            Xi[:, state_index] = fit_lasso(theta, derivatives[:, state_index], alpha)
        return Xi
    raise ValueError(f"Método de regressão desconhecido: {method}")


# Função de simulação livre: integra db/dt = Theta(b) @ Xi.
def simulate_sindy(library: PolynomialLibrary, coefficients: np.ndarray, times: np.ndarray, initial_state: np.ndarray) -> tuple[np.ndarray | None, str | None]:
    def rhs(_time: float, state: np.ndarray) -> np.ndarray:
        return library.transform_single(state) @ coefficients

    try:
        solution = solve_ivp(
            rhs,
            (float(times[0]), float(times[-1])),
            np.asarray(initial_state, dtype=np.float64),
            t_eval=np.asarray(times, dtype=np.float64),
            method="RK45",
            rtol=1e-6,
            atol=1e-8,
        )
    except Exception as exc:
        return None, str(exc)
    if not solution.success or solution.y.shape[1] != len(times):
        return None, str(solution.message)
    if not np.all(np.isfinite(solution.y)):
        return None, "Integração gerou NaN/Inf."
    return solution.y.T, None


# Função de previsão local: usa o estado real anterior para prever um passo.
def simulate_reference_guided_sindy(library: PolynomialLibrary, coefficients: np.ndarray, times: np.ndarray, reference_states: np.ndarray) -> np.ndarray:
    predicted = np.zeros_like(reference_states, dtype=np.float64)
    predicted[0] = reference_states[0]
    for index in range(1, len(times)):
        dt = float(times[index] - times[index - 1])
        rhs = library.transform_single(reference_states[index - 1]) @ coefficients
        predicted[index] = reference_states[index - 1] + dt * rhs
    return predicted


# Função textual: escreve as equações descobertas.
def format_equations(coefficients: np.ndarray, feature_names: list[str], state_symbol: str) -> str:
    lines = []
    for state_index in range(coefficients.shape[1]):
        parts = []
        for value, name in zip(coefficients[:, state_index], feature_names):
            if abs(value) <= 1e-10:
                continue
            parts.append(f"{value:+.6e}" if name == "1" else f"{value:+.6e}*{name}")
        lines.append(f"d{state_symbol}{state_index + 1}/dt = {' '.join(parts) if parts else '0.0'}")
    return "\n".join(lines)


## 5. Avaliação, busca automática e salvamento


In [ ]:
# Função de gráfico: compara trajetórias reduzidas b(t).
def plot_trajectory(times: np.ndarray, states: np.ndarray, pred: np.ndarray | None, out: Path, state_symbol: str, max_states: int) -> None:
    force_matplotlib_agg()
    n = min(max_states, states.shape[1])
    fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for i in range(n):
        axes[i].plot(times, states[:, i], label="referência", lw=2)
        if pred is not None:
            axes[i].plot(times, pred[:, i], "--", label="SINDy livre", lw=2)
        axes[i].set_ylabel(f"{state_symbol}{i+1}")
        axes[i].grid(True, alpha=0.3)
        axes[i].legend()
    axes[-1].set_xlabel("tempo")
    fig.tight_layout()
    fig.savefig(out / "trajectory_free_comparison.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


# Função de gráfico: mapa temporal dos estados reduzidos.
def plot_state_heatmap(times: np.ndarray, states: np.ndarray, out: Path, state_symbol: str) -> None:
    force_matplotlib_agg()
    fig, ax = plt.subplots(figsize=(12, 4.5))
    im = ax.imshow(states.T, aspect="auto", origin="lower", extent=[times[0], times[-1], 1, states.shape[1]], cmap="viridis")
    ax.set_xlabel("tempo")
    ax.set_ylabel(f"índice {state_symbol}")
    ax.set_title("Coordenadas SAE-POD latentes")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(out / "latent_pod_state_heatmap.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


# Função de avaliação: testa uma configuração SAE-POD-SINDy.
def evaluate_candidate(
    latent_dataset: ReducedDataset,
    latent_pod_modes: int,
    polynomial_order: int,
    use_sine: bool,
    regression_method: str,
    alpha: float,
    smooth_window: int,
) -> dict[str, Any]:
    # Passo 1: aplica POD nos latentes do SAE.
    reduced, transform = fit_latent_pod(latent_dataset, latent_pod_modes)

    # Passo 2: suaviza estados apenas para calcular derivadas/ajuste, se solicitado.
    states_for_fit = smooth_states(reduced.states, smooth_window)
    derivatives = compute_derivatives(states_for_fit, reduced.times)

    # Passo 3: cria biblioteca SINDy e ajusta Xi.
    library = PolynomialLibrary(reduced.states.shape[1], polynomial_order, use_sine, reduced.state_symbol)
    theta = library.transform(states_for_fit)
    Xi = fit_sindy(theta, derivatives, regression_method, alpha)
    derivatives_pred = theta @ Xi
    derivative_error = relative_l2(derivatives, derivatives_pred)

    # Passo 4: previsão de um passo e integração livre.
    one_step_pred = simulate_reference_guided_sindy(library, Xi, reduced.times, reduced.states)
    one_step_error = relative_l2(reduced.states, one_step_pred)
    free_pred, message = simulate_sindy(library, Xi, reduced.times, reduced.states[0])
    free_success = free_pred is not None
    free_error = None if free_pred is None else relative_l2(reduced.states, free_pred)

    # Passo 5: mede esparsidade.
    n_possible = int(Xi.size)
    n_nonzero = int(np.count_nonzero(np.abs(Xi) > 1e-10))
    sparsity = 1.0 - n_nonzero / max(n_possible, 1)

    return {
        "latent_pod_modes": int(latent_pod_modes),
        "latent_pod_energy_capture": float(reduced.metadata["latent_pod_energy_capture"]),
        "polynomial_order": int(polynomial_order),
        "use_sine": bool(use_sine),
        "regression_method": regression_method,
        "alpha": float(alpha),
        "smooth_window": int(smooth_window),
        "library_size": int(len(library.feature_names)),
        "n_possible_terms": n_possible,
        "n_nonzero_terms": n_nonzero,
        "sparsity": float(sparsity),
        "derivative_l2_error": float(derivative_error),
        "one_step_l2_error": float(one_step_error),
        "free_integration_success": bool(free_success),
        "free_trajectory_l2_error": None if free_error is None else float(free_error),
        "integration_message": message,
        "dataset": reduced,
        "transform": transform,
        "library": library,
        "Xi": Xi,
        "derivatives_reference": derivatives,
        "derivatives_pred": derivatives_pred,
        "one_step_pred": one_step_pred,
        "free_pred": free_pred,
    }


# Função de busca: escolhe preferencialmente modelos com integração livre bem-sucedida e máxima esparsidade.
def run_sae_pod_sindy_search(case_dir: Path, output_dir: Path, config: SearchConfig) -> dict[str, Any]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    latent_dataset = load_sae_latents(case_dir)
    print("\nBusca SAE-POD-SINDy")
    print(f"   Latentes: {latent_dataset.states.shape}")
    print(f"   Fonte: {latent_dataset.source}")

    candidates = []
    for modes in config.latent_pod_modes_list:
        for order in config.polynomial_orders:
            for use_sine in config.use_sine_options:
                for method in config.regression_methods:
                    for alpha in config.alpha_values:
                        for smooth_window in config.smooth_windows:
                            item = evaluate_candidate(latent_dataset, modes, order, use_sine, method, alpha, smooth_window)
                            item["accepted"] = bool(
                                item["free_integration_success"]
                                and item["derivative_l2_error"] <= config.derivative_error_limit
                                and (item["free_trajectory_l2_error"] is not None and item["free_trajectory_l2_error"] <= config.trajectory_error_limit)
                            )
                            # Score menor é melhor. Penaliza falha da integração livre fortemente.
                            free_err = 10.0 if item["free_trajectory_l2_error"] is None else item["free_trajectory_l2_error"]
                            fail_penalty = 10.0 if not item["free_integration_success"] else 0.0
                            item["score"] = float(
                                fail_penalty
                                + item["derivative_l2_error"] / max(config.derivative_error_limit, 1e-12)
                                + free_err / max(config.trajectory_error_limit, 1e-12)
                                + config.sparsity_weight * (1.0 - item["sparsity"])
                            )
                            candidates.append(item)
                            status = "OK" if item["accepted"] else "--"
                            print(
                                f"[{status}] modes={modes} order={order} sine={use_sine} {method} alpha={alpha:.1e} "
                                f"smooth={smooth_window} deriv={item['derivative_l2_error']:.4f} "
                                f"free={item['free_trajectory_l2_error']} sparsity={item['sparsity']:.3f} nnz={item['n_nonzero_terms']}"
                            )

    accepted = [c for c in candidates if c["accepted"]]
    if accepted:
        best = sorted(accepted, key=lambda c: (-c["sparsity"], c["free_trajectory_l2_error"], c["derivative_l2_error"]))[0]
        reason = "integração livre estável dentro dos limites, com máxima esparsidade"
    else:
        best = sorted(candidates, key=lambda c: c["score"])[0]
        reason = "nenhum modelo atendeu aos limites; escolhido melhor compromisso por score"

    # Salva resultados tabulares.
    fields = [
        "accepted", "score", "latent_pod_modes", "latent_pod_energy_capture", "polynomial_order", "use_sine",
        "regression_method", "alpha", "smooth_window", "library_size", "n_possible_terms", "n_nonzero_terms",
        "sparsity", "derivative_l2_error", "one_step_l2_error", "free_integration_success",
        "free_trajectory_l2_error", "integration_message",
    ]
    csv_path = output_dir / "sae_pod_sindy_search_results.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        for c in candidates:
            writer.writerow({f: c[f] for f in fields})

    # Salva equações e arrays do melhor caso.
    (output_dir / "best_equations.txt").write_text(format_equations(best["Xi"], best["library"].feature_names, best["dataset"].state_symbol), encoding="utf-8")
    np.savez_compressed(
        output_dir / "best_model.npz",
        times=best["dataset"].times,
        states=best["dataset"].states,
        Xi=best["Xi"],
        feature_names=np.asarray(best["library"].feature_names, dtype=object),
        derivatives_reference=best["derivatives_reference"],
        derivatives_pred=best["derivatives_pred"],
        one_step_pred=best["one_step_pred"],
        free_pred=np.full_like(best["dataset"].states, np.nan) if best["free_pred"] is None else best["free_pred"],
    )

    # Figuras do melhor caso.
    plot_state_heatmap(best["dataset"].times, best["dataset"].states, output_dir, best["dataset"].state_symbol)
    plot_trajectory(best["dataset"].times, best["dataset"].states, best["free_pred"], output_dir, best["dataset"].state_symbol, config.max_plot_states)

    summary = {
        "architecture": "SAE+latentPOD+SINDy",
        "selection_reason": reason,
        "n_tested": len(candidates),
        "n_accepted": len(accepted),
        "source_file": str(latent_dataset.source),
        "best": {k: make_json_ready(v) for k, v in best.items() if k not in {"dataset", "transform", "library", "Xi", "derivatives_reference", "derivatives_pred", "one_step_pred", "free_pred"}},
        "files": {
            "csv": str(csv_path),
            "best_equations": str(output_dir / "best_equations.txt"),
            "best_model": str(output_dir / "best_model.npz"),
        },
    }
    (output_dir / "summary.json").write_text(json.dumps(make_json_ready(summary), indent=2), encoding="utf-8")
    print("\nMelhor caso:")
    print(json.dumps(make_json_ready(summary["best"]), indent=2, ensure_ascii=False))
    print(f"\nResultados salvos em: {output_dir}")
    return summary


## 6. Executar busca SAE-POD-SINDy


In [ ]:
# Execute esta célula para rodar a técnica inspirada no artigo 2401.05449.
# Configuração refinada após os primeiros testes:
# - prioriza modelos lineares estáveis;
# - usa suavização maior;
# - relaxa temporariamente o erro livre para aceitar modelos integráveis e depois refinar.
config = SearchConfig(
    latent_pod_modes_list=(2, 3, 4),
    polynomial_orders=(1,),
    use_sine_options=(False,),
    regression_methods=("lasso", "stlsq"),
    alpha_values=(1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0),
    smooth_windows=(51, 101, 151, 201),
    max_plot_states=4,
    derivative_error_limit=0.90,
    trajectory_error_limit=0.60,
    sparsity_weight=0.50,
)

refined_output = OUTPUT_ROOT / "refined_linear_stable"
summary = run_sae_pod_sindy_search(CASE_DIR, refined_output, config)
summary


## 7. Resumo JSON


In [ ]:
print(json.dumps(summary, indent=2, ensure_ascii=False))
